In [ ]:
import requests
import pandas as pd
import re
import os
import time
import random
import io
from datetime import datetime, timedelta

In [19]:
# Test
res = requests.post(
    "https://traffic.dot.ga.gov/ATSPM/DefaultCharts/GetTMCMetric",
    json={
        "SignalID": "6102",
        "StartDate": "11/16/2025 12:00 AM",
        "EndDate": "11/16/2025 11:59 PM",
        "YAxisMax": "",
        "Y2AxisMax": "",
        "MetricTypeID": 5,
        "SelectedBinSize": "60",
        "ShowLaneVolumes": True,
        "ShowTotalVolumes": True,
        "ShowDataTable": True,
    }
)

print(res)
print(res.text)

<Response [200]>


In [5]:
# -----------------------------
# Config
# -----------------------------
URL = "https://traffic.dot.ga.gov/ATSPM/DefaultCharts/GetTMCMetric"

# FATAL GROUP
# SIGNAL_IDS = ["4284", "7769", "11481", "11190", "11316", "11461", "11150", "11334", "7547", "7604",
#               "4095041", "4095114", "6115052", "6115041", "8591", "161", "7714", "7113", "39", "178",
#               "4095107", "1359", "2130", "6028", "5045", "5051307", "6079",
#               "5051458", "6057045", "5051126", "7632", "7580", "1434", "7864", "7067099", "196", "7691", 
#               "8079", "1475", "7913", "7007", "8052", "7803", "1351385", "5127209",
#               "5127204", "1896", "1144", "11313", "7662", "2051", "3215274", "3215133", "3215126", 
#               "3215053", "6223049", "6668", "3246", "8043", "4200", "5285", "5011"]

# RAND GROUP
# SIGNAL_IDS = ['67337', '5216', '1315', '130', '4087', '3261001', '6257', '5127', '39015', '6313049', 
#               '7439', '3255014', '2128', '8587', '1076', '6015052', '2086', '7073', '4252', '7323', 
#               '4138', '6456', '5023', '3145', '4095103', '71006', '7218', '7055', '3215244', '391', 
#               '3215268', '7595', '5127014', '39057', '38001', '3161', '3089', '3516', '2027', '5224', 
#               '1426', '5051006', '3021185', '1175', '7157', '8613', '5051432', '195', '7060', '1351333', 
#               '3086', '3077010', '6057006', '39026', '3151024', '6129022', '1429', '426', '6254', '1847', 
#               '1351279', '7214', '3215181', '5076', '8164', '6299', '8097', '7194', '8260', '75', '7063032', 
#               '6099', '3215136', '7066', '6025', '5051519', '71031', '7505', '6369', '3021197', '8544', '5051569', 
#               '1232', '5246', '1081', '229', '4194', '3153056', '3042', '4095116', '1156', '6015051', '2245059', 
#               '7063138', '2019', '3021050', '1077', '4096', '2017', '4232']

# SIGNAL_IDS = ['3612', '6423', '11302', '7263', '5328', '11174', '1636', '8004', '7034', '4217', 
#               '7012', '2116', '6045007', '5207', '1640', '3153052', '7787', '39035', '39020', '7905']

# from jose's georgia fars list
SIGNAL_IDS = ['7730', '6455', '6366', '475', '1351341', '5025', '3553', '11148', '11470', '7847', '5063', '204', 
              '6313045', '11416', '5051004', '7067108', '7608', '3255013', '6140', '11133', '7805', '47031', '11430', 
              '7013', '7770', '11361', '7860', '1703', '6672', '8171', '11142', '1435', '3517', '6115060', '5217', '1240', 
              '3242', '1905', '4071', '1464', '2245004', '7444', '5051340', '5185', '6999', '1219031', '11320', 
              '3309', '3512', '6115086', '7166', '3021212', '8235', '7207', '5051109', '2203', '71014', '3415', 
              '1351015', '4132', '3105', '7649', '4253', '1282', '5051410', '7110', '471', '2245008', '3021063', 
              '5206', '5233', '3153020', '11509', '216', '1373', '3215061', '5316', '6115109', '7198', '5024', '11433', 
              '7870', '6057044', '6115034', '8195', '6223049', '6652', '1351371', '6301', '4082', '1113', '3285008', '7678', 
              '3505', '8052', '7118', '3319', '11498', '4108', '7097046', '6015002', '3215196', '7398', '7400', '7807', '11463', 
              '7856', '7774', '5349', '5262', '3151051', '6156', '1500', '1351369', '8582', '8034', '7456', '4217', '11351', '8284', 
              '11137', '71025', '7004', '4004', '6313067', '1124', '7067243', '6318', '82011', '5018', '59010', '7804', '6234', #iploaded up to this rpw
              '7339', '212', '5051419', '6115066', '7742', '3333', '3215218', '6313013', '3068', '7063121', '11149', '5051309', 
              '5051493', '6647', '6115069', '175', '3215171', '58009', '3021202', '4090', '5373', '7865', '7927', '7688', '6426', 
              '5051418', '7117', '5003', '8608', '2241', '47042', '3021094', '1179', '2245026', '7912', '1615', '5022', '3021013', 
              '3215253', '3165', '1062', '4109', '3215034', '1351363', '1795', '1108', '1309', '6057208', '7052', '5051117', '11060', 
              '1410', '7067186', '3215091', '134', '3215112', '7383', '3262', '7067237', '1274', '8208', '3071', '7866', '7581', 
              '3077020', '7063086', '1657']


START_DATE = "5/01/2025 12:00 AM"
END_DATE   = "11/01/2025 11:59 PM"

OUTPUT_DIR = "signal_excels"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Regex to extract TMCTable
TMCTABLE_RE = re.compile(
    r'<div class="TMCTable">(.*?)</div>',
    re.DOTALL | re.IGNORECASE
)

# -----------------------------
# Helper
# -----------------------------
def split_date_range(start_str, end_str):
    start_dt = datetime.strptime(start_str, "%m/%d/%Y %I:%M %p")
    end_dt = datetime.strptime(end_str, "%m/%d/%Y %I:%M %p")
    ranges = []
    current_start = start_dt
    while current_start < end_dt:
        current_end = min(current_start + timedelta(days=7) - timedelta(seconds=1), end_dt)
        ranges.append((current_start, current_end))
        current_start = current_end + timedelta(seconds=1)
    return ranges

# -----------------------------
# API Call
# -----------------------------
for sid in SIGNAL_IDS:
    
    # ---------------------------------------------
    # Skip if output file already exists
    # ---------------------------------------------
    output_path = os.path.join(OUTPUT_DIR, f"tmc_{sid}.xlsx")
    if os.path.exists(output_path):
        print(f"Skipping Signal {sid}: file already exists ({output_path})")
        continue
    # ---------------------------------------------
    
    print(f"\nFetching TMC for signal {sid}...")
    all_dfs = []

    for start_dt, end_dt in split_date_range(START_DATE, END_DATE):
        payload = {
            "SignalID": sid,
            "StartDate": start_dt.strftime("%m/%d/%Y %I:%M %p"),
            "EndDate": end_dt.strftime("%m/%d/%Y %I:%M %p"),
            "YAxisMax": "",
            "Y2AxisMax": "",
            "MetricTypeID": 5,
            "SelectedBinSize": "60",
            "ShowLaneVolumes": True,
            "ShowTotalVolumes": True,
            "ShowDataTable": True,
        }

        try:
            res = requests.post(URL, json=payload)
            res.raise_for_status()
            html = res.text

            # Extract TMCTable block
            match = TMCTABLE_RE.search(html)
            if not match:
                print(f"No TMCTable found for {sid} ({payload['StartDate']} - {payload['EndDate']}), skipping.")
                continue

            tmc_html = match.group(1)
            # Extract table element
            table_match = re.search(r"<table.*?>.*?</table>", tmc_html, re.DOTALL | re.IGNORECASE)
            if not table_match:
                print(f"No table found for {sid}, skipping.")
                continue

            table_html = table_match.group(0)
            df = pd.read_html(io.StringIO(table_html))[0]
            df = df.iloc[:-1]

            # Flatten multi-row headers if needed
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [
                    "_".join([str(c) for c in col if "Unnamed" not in str(c)])
                    for col in df.columns
                ]
            df.columns = [col.strip().replace(" ", "_") for col in df.columns]
            df["SignalID"] = sid
            df["Datetime"] = pd.date_range(start=start_dt, periods=len(df), freq='h')
            
            all_dfs.append(df)
            print(f"   ✓ Collected {len(df)} rows ({payload['StartDate']} - {payload['EndDate']})")

            # Random delay 1-5 seconds
            time.sleep(random.uniform(1, 5))

        except Exception as e:
            print(f"Error for {sid} ({payload['StartDate']} - {payload['EndDate']}): {e}")
            # Save partial results so far
            if all_dfs:
                partial_path = os.path.join(OUTPUT_DIR, f"tmc_{sid}_partial.xlsx")
                pd.concat(all_dfs, ignore_index=True).to_excel(partial_path, index=False)
                print(f"Saved partial data to {partial_path}")
            continue

    # Save final Excel for this signal
    if all_dfs:
        out_path = os.path.join(OUTPUT_DIR, f"tmc_{sid}.xlsx")
        pd.concat(all_dfs, ignore_index=True).to_excel(out_path, index=False)
        print(f"Saved final Excel: {out_path} ({sum(len(df) for df in all_dfs)} rows)")

print("\nDone.")



📡 Fetching TMC for signal 7730...
   ✓ Collected 168 rows (05/01/2025 12:00 AM - 05/07/2025 11:59 PM)
   ✓ Collected 168 rows (05/08/2025 12:00 AM - 05/14/2025 11:59 PM)
   ✓ Collected 168 rows (05/15/2025 12:00 AM - 05/21/2025 11:59 PM)
   ✓ Collected 168 rows (05/22/2025 12:00 AM - 05/28/2025 11:59 PM)
   ✓ Collected 168 rows (05/29/2025 12:00 AM - 06/04/2025 11:59 PM)
   ✓ Collected 168 rows (06/05/2025 12:00 AM - 06/11/2025 11:59 PM)
   ✓ Collected 168 rows (06/12/2025 12:00 AM - 06/18/2025 11:59 PM)
   ✓ Collected 168 rows (06/19/2025 12:00 AM - 06/25/2025 11:59 PM)
   ✓ Collected 168 rows (06/26/2025 12:00 AM - 07/02/2025 11:59 PM)
   ✓ Collected 168 rows (07/03/2025 12:00 AM - 07/09/2025 11:59 PM)
   ✓ Collected 168 rows (07/10/2025 12:00 AM - 07/16/2025 11:59 PM)
   ✓ Collected 168 rows (07/17/2025 12:00 AM - 07/23/2025 11:59 PM)
   ✓ Collected 168 rows (07/24/2025 12:00 AM - 07/30/2025 11:59 PM)
   ✓ Collected 168 rows (07/31/2025 12:00 AM - 08/06/2025 11:59 PM)
   ✓ Collecte